1. Mounts your Google Drive
2. Loads the already-prepped dataset (PlantDoc + SoyCotton, merged, single `leaf`
   class, cross-split duplicates removed) from a zip you've uploaded to Drive --
   see Step 3 for exactly what to upload and where
3. Trains YOLO26m as a single-class leaf detector
4. Validates the model
5. Optionally trains YOLO26s the same way afterwards, so you can compare and
   decide whether the lighter model is good enough for your use case

Why this dataset isn't fetched fresh in Colab (like the old version of this
notebook did): the original PlantDoc-only dataset had 0 cotton images and only
33 soy images out of 2205, and had ~1% of images leaking across train/val/test
(same photo saved under two different names, landing in different splits).
Both were fixed locally by merging in the SoyCotton dataset (Kellermann et al.,
Scientific Data 2026 -- 640 field images, 7221 soy + 5190 cotton leaves,
bounding-box annotated) and de-duplicating by perceptual hash. Re-deriving all
of that inside every fresh Colab session would mean re-downloading PlantDoc +
SoyCotton and re-running the hashing dedup on every run, burning your Colab
GPU-session time on CPU work for no benefit -- so we do it once locally and
just load the result here.


In [ ]:
!nvidia-smi

Sun Jul 26 14:45:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## install packages

In [ ]:
# -U matters here: Colab's preinstalled ultralytics build is often behind,
# and older releases (anything before YOLO26 support landed) don't know the
# yolo26*.pt model names exist at all -- they fail with FileNotFoundError
# instead of downloading, with no useful error message pointing at the version.
!pip install -q -U ultralytics


## load the prepped dataset

One-time setup, done from your own machine (not in Colab):

```
python scripts/add_soycotton.py     # merges SoyCotton into data/processed
python scripts/dedupe_splits.py --apply   # removes cross-split duplicates
```

Then zip `data/processed` and upload the zip to your Drive, e.g.
`MyDrive/leaf_localisation/processed_dataset.zip`. Update `DRIVE_ZIP_PATH`
below if you put it somewhere else.


In [ ]:
DRIVE_ZIP_PATH = "/content/drive/MyDrive/leaf_localisation/processed_dataset.zip"
PROCESSED = "/content/processed"

!mkdir -p {PROCESSED}
!unzip -q -n "{DRIVE_ZIP_PATH}" -d {PROCESSED}
!find {PROCESSED} -maxdepth 2 -type d


/content/processed
/content/processed/train
/content/processed/train/labels
/content/processed/train/images
/content/processed/val
/content/processed/val/labels
/content/processed/val/images
/content/processed/test
/content/processed/test/labels
/content/processed/test/images


## sanity check

In [ ]:
import os

for split in ["train", "val", "test"]:
    images_dir = os.path.join(PROCESSED, split, "images")
    labels_dir = os.path.join(PROCESSED, split, "labels")
    image_names = {os.path.splitext(f)[0] for f in os.listdir(images_dir)}
    label_names = {os.path.splitext(f)[0] for f in os.listdir(labels_dir)}
    total_boxes = sum(
        len([l for l in open(os.path.join(labels_dir, n + ".txt")).read().splitlines() if l.strip()])
        for n in label_names
    )
    print(f"[{split}] images={len(image_names)} labels={len(label_names)} boxes={total_boxes} "
          f"missing_labels={len(image_names - label_names)} missing_images={len(label_names - image_names)}")


[train] images=1969 labels=1969 boxes=13738 missing_labels=0 missing_images=0
[val] images=567 labels=567 boxes=4131 missing_labels=0 missing_images=0
[test] images=284 labels=284 boxes=2042 missing_labels=0 missing_images=0


Expect to see: train images=1969, val images=567, test images=284 (2820 total,
19911 boxes) -- matching what was verified locally after the SoyCotton merge +
dedupe. If these numbers don't match, stop and check `DRIVE_ZIP_PATH` above
before training.


## write the data.yaml Colab needs

In [ ]:
data_yaml = f"""path: {PROCESSED}
train: train/images
val: val/images
test: test/images

nc: 1
names: ['leaf']
"""
with open("/content/data.yaml", "w") as f:
    f.write(data_yaml)
print(data_yaml)


path: /content/processed
train: train/images
val: val/images
test: test/images

nc: 1
names: ['leaf']



## train YOLO26m

`batch=8` is fixed rather than autobatch (`batch=-1`): autobatch profiles
memory at *2x* imgsz to size for `multi_scale` training, which would try to
test a 2560px batch on this 15GB T4 and OOM before training even starts --
that's not a hypothetical, it's what happened on the first attempt. If you
watch `nvidia-smi` during training and see comfortable headroom, you can bump
this to 16; if you see OOM warnings, drop it to 4. `seed=42` makes runs
reproducible so a later model/hyperparameter change can be judged against a
stable baseline.


In [ ]:
!rm -rf /content/runs/train
import os
from ultralytics import YOLO

DRIVE_PROJECT = "/content/drive/MyDrive/leaf_localisation/runs"
RUN_NAME = "train_m"
last_ckpt = f"{DRIVE_PROJECT}/{RUN_NAME}/weights/last.pt"

if os.path.exists(last_ckpt):
    print(f"Found a previous run, resuming from {last_ckpt}")
    model = YOLO(last_ckpt)
    model.train(resume=True)
else:
    print("Starting a fresh run")
    model = YOLO("yolo26m.pt")
    model.train(
        data="/content/data.yaml", epochs=120, imgsz=1280, batch=8, device=0,
        patience=25, save_period=10, copy_paste=0.3, mixup=0.15, seed=42,
        project=DRIVE_PROJECT, name=RUN_NAME,
    )

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Starting a fresh run
Ultralytics 8.4.106 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, fl

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/leaf_localisation", exist_ok=True)
!cp /content/runs/train/weights/best.pt /content/drive/MyDrive/leaf_localisation/best_m.pt


## validate

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/train/weights/best.pt")
metrics = model.val(data="/content/data.yaml", plots=True)

print(f"Precision  : {metrics.box.mp:.3f}")
print(f"Recall     : {metrics.box.mr:.3f}")
print(f"mAP50      : {metrics.box.map50:.3f}")
print(f"mAP50-95   : {metrics.box.map:.3f}")


## download the trained weights

Save `best_m.pt` (already copied to Drive above) into the local project at
`runs/detect/train/weights/best.pt` -- `scripts/infer_area.py` and
`scripts/webcam_demo.py` both expect it there by default.


In [ ]:
from google.colab import files
files.download("/content/runs/train/weights/best.pt")


## optional: train YOLO26s for comparison

Once `m`'s precision/recall above looks good, it's worth training `s` on the
exact same data to see how much you're actually giving up in exchange for a
faster/lighter model -- `s` has roughly a third of `m`'s parameters, so it
trains and runs inference faster and needs less GPU/CPU, but usually costs you
some recall on harder or more varied shapes (which, after this data fix, now
includes cotton's lobed leaves and soy's compound leaflets). Run this cell,
compare its P/R/mAP against `m`'s above, and pick whichever fits your
accuracy-vs-speed needs -- e.g. `s` for the local webcam demo if `m`'s numbers
hold up closely enough on CPU-speed inference.


In [ ]:
!rm -rf /content/runs/train_s
!yolo detect train data=/content/data.yaml model=yolo26s.pt epochs=120 imgsz=1280 batch=8 device=0 \
    patience=25 save_period=10 copy_paste=0.3 mixup=0.15 seed=42 \
    project=/content/runs name=train_s


In [ ]:
!cp /content/runs/train_s/weights/best.pt /content/drive/MyDrive/leaf_localisation/best_s.pt

model_s = YOLO("/content/runs/train_s/weights/best.pt")
metrics_s = model_s.val(data="/content/data.yaml")

print("YOLO26s:")
print(f"Precision  : {metrics_s.box.mp:.3f}")
print(f"Recall     : {metrics_s.box.mr:.3f}")
print(f"mAP50      : {metrics_s.box.map50:.3f}")
print(f"mAP50-95   : {metrics_s.box.map:.3f}")

print("\nYOLO26m (from above):")
print(f"Precision  : {metrics.box.mp:.3f}")
print(f"Recall     : {metrics.box.mr:.3f}")
print(f"mAP50      : {metrics.box.map50:.3f}")
print(f"mAP50-95   : {metrics.box.map:.3f}")


## try it on a sample image

Pick whichever weights you decided to keep (`model` = m, or `model_s` = s).


In [ ]:
from google.colab import files
from PIL import Image
import io

CHOSEN_MODEL = model  # or model_s
CONF = 0.25  # lower this (e.g. 0.1) to see more/weaker boxes, raise it to see only confident ones

uploaded = files.upload()  # pick any .jpg/.png from your computer
for name, data in uploaded.items():
    img = Image.open(io.BytesIO(data)).convert("RGB")
    result = CHOSEN_MODEL.predict(img, conf=CONF, verbose=False)[0]

    print(f"\n{name}: {len(result.boxes)} leaf(es) detected")
    for i, box in enumerate(result.boxes):
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf = float(box.conf[0])
        area_px = (x2 - x1) * (y2 - y1)
        print(f"  #{i}: conf={conf:.2f}  box=({x1:.0f},{y1:.0f})-({x2:.0f},{y2:.0f})  area_px={area_px:.0f}")

    display(Image.fromarray(result.plot()[..., ::-1]))  # plot() returns BGR; flip to RGB for display
